Pre-requisites:

On VM:
manojcloudvm@instance-20260404-024439:~$ which tcpdump
which nc
which python3
/usr/bin/tcpdump
/usr/bin/nc
/usr/bin/python3

On windows machine / powershell:
 ssh -V
OpenSSH_for_Windows_9.5p2, LibreSSL 3.8.2

get the VM's external IP using `curl ifconfig.me`


Step 4: start a TCP listener on the VM = opening a port

On the VM, run this in one terminal:

`nc -lv 9090`

The command nc -lv 9090 tells Netcat to act as a server that waits for an incoming connection.
This made the VM wait for an inbound TCP connection on port 9090.

Leave it running.

This means:

nc = netcat
-l = listen
-v = verbose
9090 = TCP port



Step 5: capture packets for that port

Open a second VM terminal and run:

`sudo tcpdump -n -i any tcp port 9090`

Leave that running too.

This let us watch the TCP packets live.
This command starts a **packet sniffer** to watch network traffic in real-time. Here is the breakdown:


* **`sudo`**: Runs the command with admin privileges (required to touch the network interface).
* **`tcpdump`**: The name of the tool used to capture and analyze packets.
* **`-n`**: Shows IP addresses and port numbers as **digits** (e.g., `136.114.99.52`) instead of trying to resolve them to names (e.g., `google.com`). This makes it faster and easier to read.
* **`-i any`**: Listens on **all** network interfaces (Ethernet, Wi-Fi, loopback, etc.) at once.
* **`tcp port 9090`**: A filter that tells the tool: "Only show me **TCP** packets moving through **port 9090**." (It ignores everything else).

**In short:** It’s your "microscope" to see the SYN, SYN-ACK, and ACK packets as they arrive at your VM.

Retried the connection and captured the handshake

Again from Windows:

`ssh -p 9090 manojcloudvm@136.114.99.52`

And tcpdump showed:

142.117.26.235.40808 > 10.128.0.2.9090: Flags [S]
10.128.0.2.9090 > 142.117.26.235.40808: Flags [S.]
142.117.26.235.40808 > 10.128.0.2.9090: Flags [.]
142.117.26.235.40808 > 10.128.0.2.9090: Flags [P.]
10.128.0.2.9090 > 142.117.26.235.40808: Flags [.]

Flags [S] --> SYN
Flags [S.] --> SYN-ACK
Flags [.] --> ACK
Flags [P.] -->client started sending actual data after the connection was established.


Notes / Thoughts:
If your VM has port 9090 open:
no one connecting yet → no TCP connection
your laptop connects and handshake completes → TCP connection exists


“Opening a port allows TCP connections to be established to that port.”

Very important learning
TCP handshake is only about establishing the connection

It does not mean:

login succeeded
SSH succeeded
HTTP succeeded
TLS succeeded

It only means:

“A reliable TCP connection now exists between client and server.”

After that, higher-level protocols start:

SSH
HTTP
HTTPS/TLS
database protocols
etc.

Feature,HTTP,HTTPS / TLS
Port,80,443
Encryption,None (Plain Text),Strong (Encrypted)
Handshakes,1 (TCP only),2 (TCP + TLS)
Visibility,tcpdump sees everything,tcpdump sees gibberish

Yes — TCP is reliable because it uses:

* **ACKs** to confirm received data
* **retransmissions** if packets are lost
* **ordering** to reassemble data correctly
* **flow/congestion control** to avoid overwhelming network/receiver

If you notice packet loss, think in this order:

1. **Confirm where**

   * local host?
   * VM?
   * network path?
   * only one app/port or everything?

2. **Check basics**

   * interface errors/drops
   * CPU/memory pressure
   * overloaded app/server
   * firewall / MTU / routing issues

3. **Useful commands**

   * `ping` → basic loss/latency
   * `traceroute` / `mtr` → where loss starts
   * `ss -s` / `netstat -s` → TCP retransmits/errors
   * `ip -s link` → NIC drops/errors
   * `tcpdump` → packet-level view

4. **Common fixes**

   * reduce congestion / bandwidth saturation
   * fix bad routing/firewall rules
   * check MTU mismatch
   * scale app/server if overloaded
   * retry/failover at app level if needed

Good interview line:

**“TCP handles packet loss by retransmitting unacknowledged segments, but repeated loss usually points to a network, host, or congestion problem that should be diagnosed at the source.”**
